# Avito: гибридный поиск (BM25F + клики) + LightGBM-ранкер


### Данные и признаки:
1) Таблица из 189 212 объявлений, клики из train и тестовые запросы
2) Приводим все запросы и объявления к единому виду
3) Признаки, которые используем:
   * BM25 по заголовку и описанию
   * BM25 по символьным n-граммам
   * LSA (чтобы при не совпадении слов все равно находить близкие запросы)
   * Популярность объявления и «память» запроса
   * Совпадение локации и категории
### Модели:
1) Отбор кандидатов. Быстрая линейная схема, чтобы посмотреть весь датасет целиком, не умирая от времени работы. То есть сначал производим отбор кандидатов (топ-300) по нормированным score + весам подобранным через Optuna
2) Переранжирование. LightGBM ранжирует кандидатов по 16 признакам, модель сама находит нелинейные взаимодействия и LambdaRank ускоряет обучение

### Проверка качества:
Мы делим все запросы в train на 3 части: A - считаем параметры кликов
                                         B - обучаем на ней ранкер 
                                         Валидационная - делим на 2 (по одной подбираем веса в Optune, по второй  проверяем как подобралось)
Результат на тестовых данных: ~75%

### Найденные ошибки и исправления 
1) Утечка ответов в ранкер. Он учился на запросах, чьи клики уже входили в сигналы. Исправлено разделением по запросам (A / B / валидация).
2) Ранкер не видел «память». Добавила этот признак
3) Раскладка портила английские запросы. Добавила проверку на корректность
4) Потеря запросов с пропусками. Из-за groupby валидация сжималась до 232 запросов, теперь пропуски заполняются заранее.
5) Разные шкалы score. Ввела нормировку внутри запроса и деление n-грамм на их число.
6) Словарь не включал слова из описаний. Теперь строится по заголовкам и описаниям.
7) Долго обучалась модель, но нашла места, где можно производить однократные вычисления вместо повторных.
                                         


### Библиотеки, которые используются и какие-то константы

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ.setdefault(_v, "4")

import gc, re, socket, unicodedata, warnings
from functools import lru_cache
from collections import defaultdict, Counter
from typing import Dict, List

import numpy as np
import pandas as pd
import scipy.sparse as sp
import lightgbm as lgb
import optuna
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from nltk.stem.snowball import SnowballStemmer
from tqdm.auto import tqdm

try:
    import torch
    from transformers import AutoTokenizer, AutoModel
except ImportError:          # torch/transformers не обязательны: есть fallback (LSA)
    torch = None

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


TRAIN_PATH   = "/kaggle/input/datasets/flander5/avito-nlp/data/train.parquet"
QUERIES_PATH = "/kaggle/input/datasets/flander5/avito-nlp/data/benchmark_queries.parquet"
ITEMS_PATH   = "/kaggle/input/datasets/flander5/avito-nlp/data/benchmark_items.parquet"

SEED    = 42
TOP_K   = 300     # сколько кандидатов отдаёт 1-я стадия в LightGBM
N_VAL   = 3000    # запросов для честной валидации
N_RANK  = 6000    # запросов для обучения ранкера (не участвуют в кликовых сигналах!)
np.random.seed(SEED)

### Нормализация текста 


In [ ]:


# Раскладка: латиница -> кириллица (только нижний регистр, так как текст уже .lower())
LAT = "qwertyuiop[]asdfghjkl;'zxcvbnm,./`"
RUS = "йцукенгшщзхъфывапролджэячсмитьбю.ё"
assert len(LAT) == len(RUS)
ENG_TO_RU = str.maketrans(LAT, RUS)

# Словари заполняются по корпусу объявлений, чтобы НЕ портить нормальные английские слова ("iphone" -> "шзрщту") и чинить только реальные ошибки раскладки
LATIN_VOCAB: set = set()
RU_VOCAB: set = set()
_LAYOUT_TOKEN = re.compile(r"^[a-z\[\];',./`]+$")

def fix_layout_token(tok: str) -> str:
    if not _LAYOUT_TOKEN.match(tok):
        return tok
    letters = re.sub(r"[^a-z]", "", tok)
    if letters and letters in LATIN_VOCAB:          # проверяем, что слово нормальное
        return tok
    tr = tok.translate(ENG_TO_RU)
    if re.fullmatch(r"[а-яё]{3,}", tr) and tr.replace("ё", "е") in RU_VOCAB:
        return tr
    return tok

def normalize_text(text, fix_layout: bool = False) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text).lower()
    if fix_layout:
        text = " ".join(fix_layout_token(t) for t in text.split())
    text = text.replace("ё", "е")
    text = re.sub(r"(\d+)([a-zа-я]+)", r"\1 \2", text)      # сплитим числа и слова (256gb -> 256 gb)
    text = re.sub(r"([a-zа-я]+)(\d+)", r"\1 \2", text) 
    text = re.sub(r"[^a-zа-я0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

_stemmer = SnowballStemmer("russian")

@lru_cache(maxsize=None)
def _stem_word(w: str) -> str:
    return _stemmer.stem(w)

def stem_text(norm_text: str) -> str:
    return " ".join(_stem_word(w) for w in norm_text.split())

print(normalize_text("ghbdtn! Продам iphone13pro 256gb в идеале")) # проверяем, что нормализация работает

In [ ]:
print("Загрузка данных...")
df_items      = pd.read_parquet(ITEMS_PATH).reset_index(drop=True)
df_queries    = pd.read_parquet(QUERIES_PATH)
df_train_raw  = pd.read_parquet(TRAIN_PATH)

print("items  :", list(df_items.columns))
print("queries:", list(df_queries.columns))
print("train  :", list(df_train_raw.columns))

item_id_to_idx = {iid: i for i, iid in enumerate(df_items["item_id"].values)}
idx_to_item_id = df_items["item_id"].values

KEY = ["search_query", "search_infm_params_text", "search_location_id"]

# Только клики по объявлениям из корпуса — остальные бесполезны и для сигналов, и для валидации
df_tr = df_train_raw[df_train_raw["item_id"].isin(idx_to_item_id)].copy()
for c in ["search_query", "search_infm_params_text"]:
    df_tr[c] = df_tr[c].fillna("").astype(str)
df_tr["search_location_id"] = df_tr["search_location_id"].fillna(-1)
print(f"Кликов всего: {len(df_train_raw)}, по объявлениям из корпуса: {len(df_tr)}")

# Сплитим на 3 части 
#   val  - валидационные данные, по ним будем проверять, что не пере/недо(обучаемся)
#   B    — обучение LightGBM-ранкера (кликовые сигналы считаются БЕЗ этих запросов -> нет утечки);
#   A    — всё остальное (по этим данным строим популярность и память "запрос -> объявления")

keys = df_tr[KEY].drop_duplicates().sample(frac=1.0, random_state=SEED).reset_index(drop=True)
n_val  = min(N_VAL,  int(0.15 * len(keys)))
n_rank = min(N_RANK, int(0.30 * len(keys)))
keys["split"] = "A"
keys.loc[: n_val - 1, "split"] = "val"
keys.loc[n_val: n_val + n_rank - 1, "split"] = "B"
df_tr = df_tr.merge(keys, on=KEY, how="left")

def group_queries(df: pd.DataFrame) -> pd.DataFrame:
    g = df.groupby(KEY, sort=False).agg(
        item_id=("item_id", lambda s: list(set(s))),
        search_category=("search_category", "first"),
    ).reset_index()
    return g

df_A         = df_tr[df_tr["split"] == "A"]
df_val_g     = group_queries(df_tr[df_tr["split"] == "val"])
df_rank_g    = group_queries(df_tr[df_tr["split"] == "B"])
print(f"Клики A: {len(df_A)} | val-запросов: {len(df_val_g)} | rank-запросов: {len(df_rank_g)}")

In [ ]:
print("Нормализация объявлений...")
raw_titles = df_items["item_title_raw"].fillna("").astype(str)
raw_descs = (df_items["item_infm_params_text"].fillna("").astype(str) + " " +
             df_items["item_description_raw"].fillna("").astype(str).str.slice(0, 1500))

df_items["norm_title"] = [normalize_text(t) for t in tqdm(raw_titles, desc="title")]

# Словари для безопасного исправления раскладки
_cnt = Counter(w for t in df_items["norm_title"] for w in t.split())
LATIN_VOCAB.update(w for w, c in _cnt.items() if c >= 2 and re.fullmatch(r"[a-z]+", w))
RU_VOCAB.update(w for w, c in _cnt.items() if c >= 3 and re.fullmatch(r"[а-я]+", w))

print(f"LATIN_VOCAB={len(LATIN_VOCAB)}, RU_VOCAB={len(RU_VOCAB)}")
print(normalize_text("lbdfy iphone 13 ghbdtn", fix_layout=True))

df_items["norm_desc"]  = [normalize_text(t) for t in tqdm(raw_descs, desc="desc")]
df_items["stem_title"] = [stem_text(t) for t in tqdm(df_items["norm_title"], desc="stem title")]
df_items["stem_desc"]  = [stem_text(t) for t in tqdm(df_items["norm_desc"],  desc="stem desc")]

def prepare_queries(df_q: pd.DataFrame) -> pd.DataFrame: # нормализация и склейка текста запроса с уточняющими параметрами
    df = df_q.reset_index(drop=True).copy()
    q = df["search_query"].fillna("").astype(str) if "search_query" in df else pd.Series([""] * len(df))
    p = df["search_infm_params_text"].fillna("").astype(str) if "search_infm_params_text" in df else pd.Series([""] * len(df))
    df["norm_query"] = [normalize_text(a + " " + b, fix_layout=True) for a, b in zip(q, p)]
    df["stem_query"] = [stem_text(t) for t in df["norm_query"]]
    return df

### Кликовые сигналы


In [ ]:
SIG = {"prior": None, "memory": None} # храним популярность объявления (как часто кликали) и память (на какие объявления кликали по похожим запросам)

def build_click_signals(df_clicks: pd.DataFrame):
    """prior[item] = log1p(клики); memory[норм.запрос] = (индексы объявлений, log1p(клики))."""
    idx = df_clicks["item_id"].map(item_id_to_idx)
    counts = idx.value_counts()
    prior = np.zeros(len(df_items), dtype=np.float32)
    prior[counts.index.values.astype(int)] = np.log1p(counts.values)

    pair = (df_clicks.assign(item_idx=idx)
            .groupby(["search_query", "search_infm_params_text", "item_idx"]).size()
            .reset_index(name="cnt"))
    uq = pair[["search_query", "search_infm_params_text"]].drop_duplicates().copy()
    uq["q_norm"] = [normalize_text(a + " " + b, fix_layout=True)
                    for a, b in zip(uq["search_query"], uq["search_infm_params_text"])]
    pair = pair.merge(uq, on=["search_query", "search_infm_params_text"])
    pair = pair[pair["q_norm"] != ""]

    tmp = defaultdict(lambda: defaultdict(int))
    for qn, ii, c in zip(pair["q_norm"], pair["item_idx"], pair["cnt"]):
        tmp[qn][int(ii)] += int(c)
    memory = {qn: (np.fromiter(d.keys(), dtype=np.int64),
                   np.log1p(np.fromiter(d.values(), dtype=np.float32)))
              for qn, d in tmp.items()}
    return prior, memory

# Берем кликовые сигналы только из выборки A
SIG["prior"], SIG["memory"] = build_click_signals(df_A)
print(f"Популярность: {(SIG['prior'] > 0).sum()} объявлений; память: {len(SIG['memory'])} запросов")

In [ ]:
print("Векторизация корпуса...")

'''Тексты объявлений векторизуются в словные (заголовок, описание)
    и символьные n-граммные (заголовок) представления 
    и переводятся в BM25-веса с отдельными параметрами для заголовка и описания.
    Матрицы транспонируем заранее, чтобы score для пачки запросов считать одним умножением.
'''

all_text = pd.concat([df_items["stem_title"], df_items["stem_desc"]])
vec_word = CountVectorizer(min_df=1, max_df=0.4, token_pattern=r"(?u)\b\w+\b", dtype=np.float32)
vec_word.fit(all_text)  # словарь по заголовку и тексту объявления
vec_char = CountVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.4, dtype=np.float32)

X_title = vec_word.transform(df_items["stem_title"])
X_desc  = vec_word.transform(df_items["stem_desc"])
X_char  = vec_char.fit_transform(df_items["norm_title"])
title_len = np.asarray(X_title.sum(axis=1)).ravel().astype(np.float32)

def build_bm25(X: sp.csr_matrix, k1: float, b: float) -> sp.csr_matrix:
    N = X.shape[0]
    dfreq = np.asarray((X > 0).sum(axis=0)).ravel()
    idf = np.maximum(np.log((N - dfreq + 0.5) / (dfreq + 0.5) + 1.0), 1e-6)
    dl = np.asarray(X.sum(axis=1)).ravel()
    avg = max(dl.mean(), 1e-6)
    C = X.tocoo()
    pen = np.maximum(k1 * (1.0 - b + b * dl / avg), 1e-6)[C.row]
    data = (C.data * (k1 + 1.0)) / (C.data + pen) * idf[C.col]
    return sp.csr_matrix((data.astype(np.float32), (C.row, C.col)), shape=X.shape)

XT_title  = build_bm25(X_title, 1.5, 0.75).T.tocsr()
XT_desc   = build_bm25(X_desc,  1.2, 0.50).T.tocsr()
XT_char   = build_bm25(X_char,  1.2, 0.75).T.tocsr()
XT_btitle = (X_title > 0).astype(np.float32).T.tocsr()
del X_title, X_desc, X_char, all_text
gc.collect()
print("BM25 готов")

In [ ]:
# Dense-энкодер через LSA (char-TFIDF + SVD по заголовкам)

class LSAEncoder:
    def __init__(self, dim: int = 128):
        self.dim = dim
    def fit(self, texts):
        self.tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=5, max_df=0.4,
                                     sublinear_tf=True, dtype=np.float32)
        X = self.tfidf.fit_transform(texts)
        self.svd = TruncatedSVD(self.dim, random_state=SEED, n_iter=4).fit(X)
        return self
    def encode(self, texts: List[str], batch_size: int = 50000) -> np.ndarray:
        outs = []
        for i in range(0, len(texts), batch_size):
            Z = self.svd.transform(self.tfidf.transform(texts[i:i + batch_size])).astype(np.float32)
            Z /= np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)
            outs.append(Z)
        return np.vstack(outs)

# тут вот обучаемся
encoder = LSAEncoder().fit(df_items["norm_title"])
item_dense = encoder.encode(df_items["norm_title"].tolist())
item_dense_T = np.ascontiguousarray(item_dense.T)
print("Эмбеддинги корпуса:", item_dense.shape)

### Build

In [ ]:
FEATURE_NAMES = ["bm25_title", "bm25_desc", "char", "dense",
                 "bm25_title_n", "bm25_desc_n", "char_n", "dense_n",
                 "title_cover", "prior", "memory", "loc_match", "cat_match",
                 "q_len", "title_len", "q_has_loc"]
F = len(FEATURE_NAMES)

DEFAULT_W = dict(w_title=3.0, w_desc=1.0, w_char=1.5, w_dense=2.0, w_prior=0.5, w_mem=3.0, w_loc=1.0)

def _num(s: pd.Series, fill: float) -> np.ndarray:
    return pd.to_numeric(s, errors="coerce").fillna(fill).values.astype(np.float64)

ITEM_LOC = _num(df_items["item_location_id"], -2) if "item_location_id" in df_items else np.full(len(df_items), -2.0)
ITEM_CAT = _num(df_items["item_category_id"], -2) if "item_category_id" in df_items else np.full(len(df_items), -2.0)

def encode_queries(df: pd.DataFrame) -> dict:
    
    """Тяжёлая часть (векторизация + dense-эмбеддинг) считается один раз на набор запросов."""
    Qw = vec_word.transform(df["stem_query"]).tocsr()
    nq = np.diff(Qw.indptr).astype(np.float32)
    Qw.data[:] = 1.0                                             # бинаризация query TF
    Qc = vec_char.transform(df["norm_query"]).tocsr()
    Qc.data[:] = 1.0
    nc = np.maximum(np.diff(Qc.indptr), 1).astype(np.float32)
    Qc = (sp.diags(1.0 / nc) @ Qc).tocsr().astype(np.float32)   # нормировка на число n-грамм
    return dict(
        Qw=Qw, Qc=Qc, nq=nq,
        Qd=encoder.encode(df["norm_query"].tolist()),
        loc=_num(df["search_location_id"], -1) if "search_location_id" in df else np.full(len(df), -1.0),
        cat=_num(df["search_category"], -1) if "search_category" in df else np.full(len(df), -1.0),
        qnorm=df["norm_query"].values,
    )

def minmax(M: np.ndarray) -> np.ndarray: # нормировка score к [0;1]
    lo = M.min(axis=1, keepdims=True)
    rng = np.maximum(M.max(axis=1, keepdims=True) - lo, 1e-6)
    return (M - lo) / rng

def stage1(blk: dict, weights: Dict[str, float], top_k: int = TOP_K, batch: int = 128):
    
    """в stage1 считается метрика по: BM25(title+desc+char) + dense + популярность + память + локация
    Возвращает (cand[n,top_k], feats[n,top_k,F]) — кандидаты отсортированы по скору 1-й стадии."""
    
    n, N = blk["Qw"].shape[0], len(df_items)
    prior_raw = SIG["prior"]
    prior_n = prior_raw / max(float(prior_raw.max()), 1e-6)
    memory = SIG["memory"]
    cand  = np.empty((n, top_k), dtype=np.int32)
    feats = np.empty((n, top_k, F), dtype=np.float32)

    for s in range(0, n, batch):
        e = min(s + batch, n); b = e - s
        qw, qc, qd = blk["Qw"][s:e], blk["Qc"][s:e], blk["Qd"][s:e]
        nq = blk["nq"][s:e]

        s_title = (qw @ XT_title).toarray()
        s_desc  = (qw @ XT_desc).toarray()
        s_char  = (qc @ XT_char).toarray()
        s_dense = qd @ item_dense_T
        s_cov   = (qw @ XT_btitle).toarray() / np.maximum(nq, 1.0)[:, None]

        n_title, n_desc, n_char, n_dense = map(minmax, (s_title, s_desc, s_char, s_dense))

        q_loc = blk["loc"][s:e]
        loc_match = (ITEM_LOC[None, :] == q_loc[:, None]) & (q_loc[:, None] >= 0)
        mem = np.zeros((b, N), dtype=np.float32)
        for j in range(b):
            entry = memory.get(blk["qnorm"][s + j])
            if entry is not None:
                mem[j, entry[0]] = entry[1]

        comb = (weights["w_title"] * n_title + weights["w_desc"] * n_desc +
                weights["w_char"] * n_char + weights["w_dense"] * n_dense +
                weights["w_prior"] * prior_n[None, :] + weights["w_mem"] * mem +
                weights["w_loc"] * loc_match)

        part = np.argpartition(comb, -top_k, axis=1)[:, -top_k:]
        order = np.argsort(-np.take_along_axis(comb, part, 1), axis=1)
        top = np.take_along_axis(part, order, 1)
        tk = lambda M: np.take_along_axis(M, top, 1).astype(np.float32)
        cat_match = ((ITEM_CAT[top] == blk["cat"][s:e][:, None]) & (blk["cat"][s:e][:, None] >= 0))

        cols = [tk(s_title), tk(s_desc), tk(s_char), tk(s_dense),
                tk(n_title), tk(n_desc), tk(n_char), tk(n_dense),
                tk(s_cov), prior_raw[top], tk(mem), loc_match[np.arange(b)[:, None], top].astype(np.float32),
                cat_match.astype(np.float32),
                np.broadcast_to(nq[:, None], top.shape), title_len[top],
                np.broadcast_to((q_loc >= 0).astype(np.float32)[:, None], top.shape)]
        feats[s:e] = np.stack(cols, axis=-1)
        cand[s:e] = top
        del s_title, s_desc, s_char, s_dense, s_cov, n_title, n_desc, n_char, n_dense, mem, comb, loc_match
    return cand, feats

def true_idx(ids) -> np.ndarray:
    return np.array([item_id_to_idx[i] for i in ids if i in item_id_to_idx], dtype=np.int64)

def recall_at(df_g: pd.DataFrame, order_cand: np.ndarray, k: int) -> float: # метрика recall-k
    r = []
    for i, ids in enumerate(df_g["item_id"]):
        t = true_idx(ids)
        if len(t):
            r.append(np.isin(order_cand[i, :k], t).sum() / len(t))
    return float(np.mean(r))

In [ ]:
print("Подготовка запросов...")
df_rank_p = prepare_queries(df_rank_g)
df_val_p  = prepare_queries(df_val_g)

blk_rank = encode_queries(df_rank_p)
print("Кандидаты для обучения ранкера...")
cand_rank, feats_rank = stage1(blk_rank, DEFAULT_W)

'''LightGBM переупорядочивает 300 кандидатов каждого запроса по 16 признакам, 
    чтобы кликнутые объявления оказались выше'''

def make_rank_dataset(df_g, cand, feats):
    """Собирает обучающую выборку для ранкера: метка 1, если по кандидату был клик.
    Запросы без кликов среди кандидатов пропускаются только на обучении."""
    Xs, ys, gs = [], [], []
    for i, ids in enumerate(df_g["item_id"]):
        y = np.isin(cand[i], true_idx(ids)).astype(np.int32)
        if y.sum() == 0:      # нет ни одного клика среди кандидатов: ранкеру не на чем учиться
            continue
        Xs.append(feats[i]); ys.append(y); gs.append(len(y))
    # если доля низкая, проблема в stage1 (отборе кандидатов), а не в ранкере
    print(f"Запросов для обучения: {len(gs)} из {len(df_g)} (клик попал в топ-{cand.shape[1]})")
    return np.vstack(Xs), np.concatenate(ys), gs

X_all, y_all, g_all = make_rank_dataset(df_rank_p, cand_rank, feats_rank)

# query-wise разделение на train / early-stopping
n_g = len(g_all); n_tr = int(0.85 * n_g)
cut = int(np.sum(g_all[:n_tr]))
ranker = lgb.LGBMRanker(objective="lambdarank", n_estimators=600, learning_rate=0.05, num_leaves=31,
                        min_child_samples=20, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                        lambdarank_truncation_level=60, random_state=SEED, n_jobs=4, verbose=-1)
ranker.fit(X_all[:cut], y_all[:cut], group=g_all[:n_tr],
           eval_set=[(X_all[cut:], y_all[cut:])], eval_group=[g_all[n_tr:]], eval_at=[50],
           callbacks=[lgb.early_stopping(30, verbose=False)])
print("Ranker обучен, best_iteration =", getattr(ranker, "best_iteration_", None))
print(dict(zip(FEATURE_NAMES, ranker.feature_importances_)))

def rerank(cand, feats):
    sc = ranker.predict(feats.reshape(-1, F)).reshape(cand.shape)
    return np.take_along_axis(cand, np.argsort(-sc, axis=1), 1)

### Валидация

In [ ]:
n_tune = min(1000, len(df_val_p) // 2)
df_tune, df_hold = df_val_p.iloc[:n_tune].reset_index(drop=True), df_val_p.iloc[n_tune:].reset_index(drop=True)
blk_tune, blk_hold = encode_queries(df_tune), encode_queries(df_hold)

def evaluate(df_g, blk, weights, verbose=True):
    cand, feats = stage1(blk, weights)
    res = {"stage1@50": recall_at(df_g, cand, 50),
           "stage1@100": recall_at(df_g, cand, 100),
           f"stage1@{cand.shape[1]}": recall_at(df_g, cand, cand.shape[1]),
           "reranked@50": recall_at(df_g, rerank(cand, feats), 50)}
    if verbose:
        for k, v in res.items():
            print(f"  {k:>14}: {v:.4f}")
    return res

print(f"Hold-out ({len(df_hold)} запросов), веса по умолчанию:")
res_default = evaluate(df_hold, blk_hold, DEFAULT_W)

## Считаем answer

In [ ]:
def objective(trial):
    w = dict(
        w_title=trial.suggest_float("w_title", 0.5, 5.0),
        w_desc =trial.suggest_float("w_desc", 0.0, 2.0),
        w_char =trial.suggest_float("w_char", 0.0, 3.0),
        w_dense=trial.suggest_float("w_dense", 0.0, 4.0),
        w_prior=trial.suggest_float("w_prior", 0.0, 1.5),
        w_mem  =trial.suggest_float("w_mem", 0.0, 8.0),
        w_loc  =trial.suggest_float("w_loc", 0.0, 3.0),
    )
    cand, feats = stage1(blk_tune, w)
    return recall_at(df_tune, rerank(cand, feats), 50)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.enqueue_trial(DEFAULT_W)
study.optimize(objective, n_trials=15) #рандомно выбираем веса 10 раз и еще 5 улучшаем по полученным данным
print("Лучшие веса:", {k: round(v, 3) for k, v in study.best_params.items()})
print(f"Recall@50 на tune-части: {study.best_value:.4f}")

print(f"\nHold-out с подобранными весами:")
res_best = evaluate(df_hold, blk_hold, study.best_params)

# Берём подобранные веса только если они лучше на hold-out (Optuna может переобучиться на tune)
use_best = res_best["reranked@50"] >= res_default["reranked@50"]
FINAL_W  = dict(study.best_params) if use_best else dict(DEFAULT_W)
chosen   = res_best if use_best else res_default
USE_RERANK = chosen["reranked@50"] >= chosen["stage1@50"]      # если LightGBM не помог, то оставляем stage1
print("FINAL_W:", {k: round(v, 3) for k, v in FINAL_W.items()}, "| rerank:", USE_RERANK)

In [ ]:
# Фаза 2: пересчитываем клики по всему train, остальное оставляем из фазы 1
SIG["prior"], SIG["memory"] = build_click_signals(df_tr)

df_test_p = prepare_queries(df_queries)
blk_test = encode_queries(df_test_p)
cand_test, feats_test = stage1(blk_test, FINAL_W)
final_idx = rerank(cand_test, feats_test)[:, :50] if USE_RERANK else cand_test[:, :50]

answer = pd.DataFrame({
    "query_id": df_test_p["query_id"],
    "answer": [" ".join(map(str, idx_to_item_id[row])) for row in final_idx],
})
assert len(answer) == len(df_queries) and answer["answer"].str.split().str.len().eq(50).all()
answer.to_csv("answer.csv", index=False)
print("answer.csv сохранён уи-уи-уи")
answer.head()